<a href="https://colab.research.google.com/github/imanuni/imanuni/blob/main/ARABERT-TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch -q

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:

from google.colab import files

# Sélectionne ton fichier local
uploaded = files.upload()
import os
print(os.listdir())  # Vérifie que model.xlsx est bien là
df = pd.read_csv("model-clean.csv", encoding="utf-8")
print(" Données brutes :")
print(df.head())

Saving model-clean.csv to model-clean.csv
['.config', 'model-clean.csv', 'sample_data']
📌 Données brutes :
                                               titre label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف   yes
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...    no
2                 تفاؤل بحل قضية الودائع: على أي أسس    no
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...    no
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...    no


In [ ]:
df["label"] = df["label"].map({"yes": 1, "no": 0})

print("\n Après conversion yes/no → 0/1 :")
print(df.head())


 Après conversion yes/no → 0/1 :
                                               titre  label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف      1
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...      0
2                 تفاؤل بحل قضية الودائع: على أي أسس      0
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...      0
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...      0


In [ ]:
 #Création Dataset HuggingFace

dataset = Dataset.from_pandas(df)

print("\n Exemple dataset brut (5 lignes) :")
print(dataset[:5])


 Exemple dataset brut (5 lignes) :

 {'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0]}


In [ ]:
 #Tokenisation avec AraBERT

model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["titre"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = dataset.map(tokenize_function, batched=True)

print("\n Exemple dataset après tokenisation (5 lignes) :")
print(dataset[:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/1113 [00:00<?, ? examples/s]


📌 Exemple dataset après tokenisation (5 lignes) :
{'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0], 'input_ids': [[2, 6393, 31, 14494, 3228, 1995, 3772, 30663, 49599, 28698, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2764, 22296, 27705, 578, 31, 3249, 6520, 33828, 667, 23698, 197, 1211, 1381, 1350, 1719, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 29188, 10694, 2205, 15542, 31, 323, 559, 6910, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
# Conversion de label en ClassLabel
from datasets import Dataset, ClassLabel
class_labels = ClassLabel(num_classes=2, names=["autre", "declaration"])
dataset = dataset.cast_column("label", class_labels)

# Split train/test avec stratification
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="label")

print("\n Train set (5 lignes) :")
print(dataset["train"][:5])


Casting the dataset:   0%|          | 0/1113 [00:00<?, ? examples/s]


📌 Train set (5 lignes) :
{'titre': ['ترامب: نيسان يوم تحرير الولايات المتحدة', 'وزير الداخلية: لا تهاون في متابعة استحقاق الانتخابات البلدية', 'سيلرغراف لـالنهار: هناك ضرورة قانونية لحظر حزب الله', 'بري: الموازين لم تتغير', 'العدو يوسع خطوط الانسحاب: مناورة تفاوضية تحت النار'], 'label': [1, 1, 1, 1, 0], 'input_ids': [[2, 6297, 31, 6024, 759, 3690, 1405, 848, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 902, 1542, 31, 391, 43131, 305, 4348, 19932, 1449, 3039, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 5205, 183, 33458, 1, 31, 780, 2096, 7421, 46263, 1460, 647, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3225, 31, 44992, 407, 1

In [ ]:
#Tokenisation avec AraBERT
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["titre"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.map(tokenize_function, batched=True)
print(" Exemple après tokenisation :")
print(dataset[:5])



Map:   0%|          | 0/1113 [00:00<?, ? examples/s]

Map:   0%|          | 0/1113 [00:00<?, ? examples/s]

📌 Exemple après tokenisation :
{'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0], 'input_ids': [[2, 6393, 31, 14494, 3228, 1995, 3772, 30663, 49599, 28698, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2764, 22296, 27705, 578, 31, 3249, 6520, 33828, 667, 23698, 197, 1211, 1381, 1350, 1719, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 29188, 10694, 2205, 15542, 31, 323, 559, 6910, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
# Conversion yes -> 1, no -> 0
df["label"] = df["label"].map({"yes": 1, "no": 0})

print(df.head())

                                               titre  label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف    NaN
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...    NaN
2                 تفاؤل بحل قضية الودائع: على أي أسس    NaN
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...    NaN
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...    NaN


In [ ]:
# Charger le modèle
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#Définir métriques

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        p.label_ids, preds, average="binary"
    )
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
import transformers
print(transformers.__version__)

4.56.1


In [ ]:
print(type(dataset))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['titre', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 890
    })
    test: Dataset({
        features: ['titre', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 223
    })
})


In [ ]:
  #Tokenisation
  tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

  def tokenize_function(examples):
      return tokenizer(examples["titre"], padding="max_length", truncation=True)

  tokenized_datasets = dataset.map(tokenize_function, batched=True)

  train_dataset = tokenized_datasets["train"]
  test_dataset  = tokenized_datasets["test"]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/890 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [ ]:
#  Charger le modèle
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased", num_labels=2
)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#  Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-1819497024.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:

# Définir les métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
#  Lancer l’entraînement
trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: